# Liver Tumor Segmentation with Synthetic Data

A lesion inserted into a CT scan comes with a known location and shape. That gives a segmentation network both a training image and a voxel-level target. For my Neural Networks course, I implemented this approach for liver CT using MONAI and PyTorch Lightning, adapting the synthesis method of [Hu et al., *Label-Free Liver Tumor Segmentation*, CVPR 2023](https://openaccess.thecvf.com/content/CVPR2023/html/Hu_Label-Free_Liver_Tumor_Segmentation_CVPR_2023_paper.html).

## CT volumes and annotations

LiTS uses three voxel labels: 0 for background, 1 for liver and 2 for tumor. For synthesis, the input is a healthy CT and its liver mask. For evaluation, the annotation also identifies real lesions.

The reference manifest selects 116 healthy training cases from BTCV, TCIA, CHAOS and LiTS. Select `lits_only` to use the 11 healthy LiTS cases. Both protocols use the same 22 LiTS validation cases. Select `holdout` for a separate evaluation on the remaining 98 LiTS cases.

In [ ]:
from pathlib import Path
import torch
from lightning.pytorch import seed_everything
from monai.utils import set_determinism
from syntumor.data import SyntheticTumorDataModule, build_splits
from syntumor.model import UNet3DLightning

SEED = 420
DATA_ROOT = Path("data")
PROTOCOL = "reference"  # "lits_only" only if you intentionally use the reduced dataset
EVALUATION_SPLIT = "validation"  # or "holdout", explicitly separate from reference evaluation
RUN_DIR = Path("runs") / f"corrected_{PROTOCOL}"
NUM_WORKERS = 0  # Portable notebook setting; increase on a suitable training machine.
CACHE_NUM = 200  # Set 0 if RAM cannot hold the resampled healthy cohort.
MAX_EPOCHS = 4000
VAL_EVERY = 200

seed_everything(SEED, workers=True)
set_determinism(seed=SEED)
ACCELERATOR = "gpu" if torch.cuda.is_available() else "cpu"
splits = build_splits(DATA_ROOT, PROTOCOL, check_files=True)
print({name: len(rows) for name, rows in splits.items()})

## Online tumor synthesis

![Tumor synthesis diagram showing location selection, texture and shape generation, and placement in a liver CT](assets/tumor-synthesis.png)

*Figure from [Hu et al., *Label-Free Liver Tumor Segmentation*, CVPR 2023](https://openaccess.thecvf.com/content/CVPR2023/html/Hu_Label-Free_Liver_Tumor_Segmentation_CVPR_2023_paper.html), reproduced from the course notebook. White boxes mark the steps used for model training. Blue boxes show additional effects used for the paper’s visual Turing test.*

The generator selects locations inside the annotated liver and deforms ellipsoids to create lesions of different sizes. It smooths their edges and uses a spatially varying texture to reduce CT intensity inside each lesion. The same geometry defines the tumor mask, so every insertion produces a paired image and annotation.

Images and labels are first reoriented to RAS and resampled to 1 mm spacing. The generator inserts lesions with probability 0.9. Intensities are then clipped to [−21, 189] HU and scaled to [0, 1]. Random 96³ crops sample foreground and background centers with equal probability. Flips, rotations and intensity changes vary the training patches.

The ten reusable textures occupy about 3.2 GB on disk. Set `CACHE_NUM=0` to disable caching of preprocessed volumes.

In [ ]:
data_module = SyntheticTumorDataModule(
    data_root=DATA_ROOT, protocol=PROTOCOL, batch_size=2,
    num_workers=NUM_WORKERS, cache_num=CACHE_NUM,
    evaluation_split=EVALUATION_SPLIT,
)
model = UNet3DLightning(max_epochs=MAX_EPOCHS, warmup_epochs=100,
                        prediction_dir=str(RUN_DIR / EVALUATION_SPLIT / "predictions"))

## Training the 3D U-Net

The 3D U-Net reads a 96 × 96 × 96 voxel patch and assigns each voxel to background, liver or tumor. Its encoder collects context across the patch. Skip connections carry fine spatial detail back to the decoder as it reconstructs the segmentation. The training objective combines Dice loss, which measures overlap, with cross-entropy, which rewards assigning the correct class to each voxel.

The configuration uses AdamW with a learning rate of 0.0004, 100 warmup epochs and cosine decay. Every 200 epochs, the validation step computes separate liver and tumor Dice scores. Their mean determines which checkpoint to retain.

In [ ]:
from lightning.pytorch import Trainer
from lightning.pytorch.callbacks import ModelCheckpoint
from lightning.pytorch.loggers import TensorBoardLogger

checkpoint_callback = ModelCheckpoint(
    monitor="val_mean_dice", mode="max", save_top_k=1, save_last=True,
    save_weights_only=False, dirpath=RUN_DIR / "checkpoints",
    filename="best-{epoch:04d}-{val_mean_dice:.4f}",
)
trainer = Trainer(
    max_epochs=MAX_EPOCHS, accelerator=ACCELERATOR, devices=1, precision="32-true",
    accumulate_grad_batches=1, check_val_every_n_epoch=VAL_EVERY,
    num_sanity_val_steps=0, log_every_n_steps=1,
    logger=TensorBoardLogger(str(RUN_DIR), name="logs"),
    callbacks=[checkpoint_callback],
)

In [ ]:
trainer.fit(model, datamodule=data_module)
# To resume from a checkpoint:
# trainer.fit(model, datamodule=data_module, ckpt_path=RUN_DIR / "checkpoints" / "last.ckpt")

## Whole-volume evaluation

During inference, overlapping windows cover the full scan. Gaussian blending gives more weight to predictions near the center of each window. The pipeline maps the combined scores back to the original CT grid before selecting a class for each voxel.

Dice measures how much the predicted and annotated volumes overlap. Normalized surface Dice measures how much of their surface area falls within 2 mm of the other boundary. Both are recorded separately for liver and tumor.

Per-case scores are written to CSV and predicted label maps are saved as NIfTI volumes with the original affine.

In [ ]:
best_path = checkpoint_callback.best_model_path
if not best_path:
    raise RuntimeError("No validated checkpoint. Train through a validation epoch first.")
print("Evaluating:", best_path, "on", EVALUATION_SPLIT)
results = trainer.test(datamodule=data_module, ckpt_path=best_path)
results